In [66]:
import datetime
import numpy as np
import quandl
quandl.ApiConfig.api_key = 'tEsTkEy123456789'
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import pandas_datareader.data as web
import pprint
import statsmodels.tsa.stattools as ts
import statsmodels.api as sm

# 均值回复
#关键的定量交易概念：指股票价格、房产价格等社会现象、自然现象（气温、降水），
#无论高于或低于价值中枢（或均值）都会以很高的概率向价值中枢回归的趋势。
#时间序列可以当价格序列很低的时候进场利用这些策略产生交易策略在期望系列将返回一个平均值，从而在市场上获利。均值回复策略是统计套利的重要部分

In [67]:
#随机游走(random walk)
#随机游走是指一个时间序列下一个定向运动完全独立于任何过去的运动————实质是时间序列没有记忆
#dx(i) = dW(i)  dW(i)~~~N(0,dt)
#然而，平均回复时间序列不同。下一个时间段的时间序列值的变化正比于当前值。在数学上这样连续的时间序列被称为Ornstein-Uhlenbeck过程
#dx(i)O(u-x(i)di +covdW(i)

In [68]:
#Augmented Dickey-Fuller(ADF)Test-检验单位根
#ADF用了这样的一个事实，即如果价格序列具有均值回复，那么下一个价格水平将于当前价格水平程反比。

In [2]:
symbol = 'WIKI/AMZN' # or 'AAPL.US'
amzn =  quandl.get(symbol,start_date='2005-01-01', end_date='2015-01-01').sort_index()
ts.adfuller(amzn['Close'],1)

(-0.5026540640718633,
 0.8914975322764598,
 1,
 2515,
 {'1%': -3.4329527780962255,
  '5%': -2.8626898965523724,
  '10%': -2.567382133955709},
 13867.020604373796)

In [19]:
#不拒绝原假设，并不是均值回复，如正常概念一样，几乎所有金融数据满是非均值回复，例如几何布朗运动

# 平稳性

In [20]:
#Hurst Exponent 可以帮助检验时间序列的平稳性
#H<0.5 时间序列是平均回复
#H = 0.5 时间序列是布朗运动
#H>0.5 时间序列具有趋势

In [10]:
from numpy import cumsum,log,polyfit,sqrt,std,subtract
from numpy.random import randn
def hurst(ts):
    lags = range(2,100)
    tau = [sqrt(std(subtract(ts[lag:],ts[:-lag]))) for lag in lags]
    #Use a linear fit to estimate the Burst Exponent 使用线性拟合来估计爆发指数
    poly = polyfit(log(lags),log(tau),1)
    #Return the Hurst exponent from the polyfit output 从polyfit的输出中返回Hurst指数
    return poly[0]*2
#Create a Gometric Brownian Motion,Mean-Reverting and Trending Series 创建一个几何布朗运动、均值回归和趋势系列
gbm = log(cumsum(randn(100000)) + 1000)
mr = log(randn(100000) + 1000)
tr = log(cumsum(randn(100000)+1)+1000)
# 输出上述每个时间序列的Hurst指数
# 以及亚马逊的价格（调整后收盘价）用于文章中上面提到的ADF检验
print('Hurst(GBM): %s' % hurst(gbm))
print('Hurst(MR): %s' % hurst(mr))
print('Hurst(TR): %s' % hurst(tr))

Hurst(GBM): 0.5003983993625537
Hurst(MR): 2.5899988630692526e-05
Hurst(TR): 0.95541990442739


In [11]:
print('Hurst(AMZN): %s' % hurst(amzn['Close']))

Hurst(AMZN): nan


C:\Users\86157\AppData\Local\Temp\ipykernel_16368\763379159.py:7: RuntimeWarning: divide by zero encountered in log
  poly = polyfit(log(lags),log(tau),1)


# Cointegration 协整-配对交易
实际上很难找到具有均值回复的可交易资产。股票广义上与GBM几何布朗相似，因此平均回复策略几乎失效。但是，依旧可以创建一个价格系列固定组合。因此，我们可以将平均回复交易策略应用于投资组合。平均回复交易策略的最简单形式是通常的经典双交易。理论上，在两家公司同一部门可能面临类似的市场因素。偶尔他们的相对股价会因为某些事情发生分歧，但是很快恢复原来的状态。
用AMEX VS WLL来寻找配对的可能
y(t) = bx(t)+c(t) c(t)是平稳均值回复

In [27]:
#画价格序列，散点图，残差序列
def plot_price_series(df,ts1,ts2):
    months = mdates.MonthLocator()
    fig,ax = plt.subplots()
    ax.plot(df.index,df[ts1],label=ts1)
    ax.plot(df.index,df[ts2],label=ts2)
    ax.xaxis.set_major_locator(months)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.set_xlim(datetime.datetime(2012,1,1),datetime.datetime(2013,1,1))
    ax.grid(True)
    fig.autofmt_xdate()
    plt.xlabel('Month/Year')
    plt.ylabel('Price($)')
    plt.title(' %s and %s Daily Prices' %(ts1,ts2))
    plt.legend()
    plt.show()
def plot_scatter_series(df,ts1,ts2):
    plt.xlabel('%s Price($)'% ts1)
    plt.ylabel('%s Price($)'% ts2)
    plt.title('%s and %s Price Scatterplot'%(ts1,ts2))
    plt.scatter(df[ts1],df[ts2])
    plt.show()
def plot_residuals(df):
    months = mdates.MonthLocator()
    fig,ax = plt.subplots()
    ax.xaxis.set_major_locator(months)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.set_xlim(datetime.datetime(2012,1,1),datetime.datetime(2013,1,1))
    ax.grid(True)
    fig.autofmt_xdate()
    plt.xlabel('Month/Year')
    plt.ylabel('Price($)')
    plt.title('Residual Plot')
    plt.legend()
    plt.plot(df['res'])
    plt.show()

In [69]:
if __name__ == '__main__':
    start = '2012-01-01'
    end = '2013-01-01'
    arex = quandl.get('WIKI/AREX',start_date = start,end_date= end).sort_index()
    wll = quandl.get('WIKI/WLL',start_date = start,end_date= end).sort_index()
    df = pd.DataFrame(index=arex.index)
    df
    df['WIKI/AREX'] = arex['Close']
    df['WIKI/WLL'] = wll['Close']
    plot_price_series(df,'WIKI/AREX','WIKI/WLL')
    plot_scatter_series(df,'WIKI/AREX','WIKI/WLL')
    y = df['WLL']
    x = df['AREX']
    res = sm.OLS(y,x).fit()
    beta_hr = res.params.AREX
    df['res'] = df['WLL'] - beta_hr * df['AREX']
    plot_residuals(df)
    cadf = ts.adfuller(df['res'])
    pprint.pprint(cadf)

LimitExceededError: (Status 429) (Quandl Error QELx06) You have exceeded the API speed limit and your account has temporaly been disabled.  Please contact clientsuccess@nasdaq.com for more information.

# 基本预测手法
算法来帮助我们预测金融时间序列的市场方向。用Scikit-learn，一种统计机器学习python的库。Scikit-learn包含许多机器学习的现成技术。
未来评估分类器的性能可以用命中率和混乱矩阵
混乱矩阵通过确定假阳性率来表征这个想法和假阴形率用于监督分类器。
逻辑斯蒂回归，朴素贝叶斯，支持向量机，决策树，随机树

In [53]:
#机器学习
import sklearn
from sklearn import svm
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.svm import LinearSVC,SVC

In [34]:
def create_lagged_series(symbol,start_date,end_date,lags=5):
    ts = quandl.get(symbol, start_date=start_date - datetime.timedelta(days=365), end_date=end_date).sort_index()
    #创建新的滞后数据框
    tslag = pd.DataFrame(index=ts.index)
    tslag['Today'] = ts['Close']
    tslag['Volume'] = ts['Volume']
    #创建先前交易期收盘价的滞后系列
    for i in range(0,lags):
        tslag['Lag%s' % str(i+1)] = ts['Clsoe'].shift(i+1)
        tsret = pd.DataFrame(index=tslag.index)
        tsret['Volume'] = tslag['Volume']
        tsret['Today'] = ts['Volume'].pct_change() * 100
    for i,x in enumerate(tsret['Today']):
        if (abs(x) < 0.0001):
            tsret['Today'][i] = 0.0001
    #创建滞后百分比收益
    for i in range(0,lags):
        tsret['Lag%s' % str(i+1)] = \
        tslag['Lag%s' % str(i+1)].pct_change()*100
    #创建一个Direction’列（+1或-1），表示上涨/下跌的交易日
    tsret['Diretion'] = np.sign(tsret['Today'])
    tsret = tsret[tsret.index >= start_date]
    return tsret

In [60]:
if __name__ == '__main__':
    #creat a lagged series of  the S&P500 US stock market index 创建S&P 500美国股票市场指数的时间序列
    snpret = create_lagged_series('WIKI/AAPL',datetime.datetime(2001,1,10),datetime.datetime(2005,12,31),lags=5)
    #use the prior two days of return s as predictor 使用前两天的收益作为预测因子
    #values，with direction as the response 将前两天的收益作为预测值，以涨跌方向作为响应。
    x = snpret[['Lag1','Lag2']]
    y = snpret['Direction']
    #the test data is split into two parts:Before and after 1st Jan 2005 测试数据分为两部分：2005年1月1日之前和之后。
    start_test = datetime.datetime(2005,1,1)
    #create training and test sets 创建训练集和测试集
    x_train = x[x.index < start_test]
    x_test = x[x.index >= start_test]
    y_train = y[y.index < start_test]
    y_test = y[y.index >= start_test]
    #create the(parametrised) models  创建参数模型
    print('Hit Rates/Confusion Matrices:\n')
    models = [('LR',LogisticRegression()),('LDA',LDA()),('QDA',QDA()),('LSVC',LinearSVC()),('RSVM',SVC(
    C=1000000,degree=3,gamma=0.0001,kernel='rbf',max_iter=-1,probability=False,random_state=None,shrinking=True,tol=0.001,verbose=False)),
             ('RF',RandomForestClassifier(n_estimators=1000,criterion='gini',max_depth=None,min_samples_split=2,
               min_samples_leaf=1,max_features='auto',bootstrap=True,oob_score=False, n_jobs=1,random_state=None,verbose=0))]
    #Iterate through the models 遍历模型
    for m in models:
        #Train each of the models on the training set
        m[1].fit(x_train,y_train)
        #make an array of predictions on the test set
        pred = m[1].predict(x_test)
        #output the hit-rate and the confusion matrix for each model
        print('%s:\n%0.3f' % (m[0],m[1].score(x_test,y_test)))
        print('%s\n' % confusion_matrix(pred,y_test))

LimitExceededError: (Status 429) (Quandl Error QELx06) You have exceeded the API speed limit and your account has temporaly been disabled.  Please contact clientsuccess@nasdaq.com for more information.

In [44]:
sklearn.linear_model.LogisticRegression(penalty='12',dual=False,tol=0.0001,C=1,fit_intercept=True,intercept_scaling=1,class_weight=None,random_state=None,
                                       solver='libinear',max_iter=100,multi_class='ovr',verbose=0,warm_start=False,n_jobs=1)

LogisticRegression(C=1, multi_class='ovr', n_jobs=1, penalty='12',
                   solver='libinear')

In [45]:
sklearn.discriminant_analysis.LinearDiscriminantAnalysis(solver='svd',shrinkage=None,priors=None,n_components=None,store_covariance=False,tol=0.0001)

LinearDiscriminantAnalysis()

In [47]:
Solver to use,possible values:
    'svd':Singular value decomposition(default).Does notcompute the covariance matrix,therefore this solver is recomm

SyntaxError: invalid syntax (12098200.py, line 1)

In [49]:
#分类
SVC(
    C=1000000,cache_size=200,class_weight=None,coef0=0,degree=3,gamma=0.0001,kernel='rbf',shrinking=True,tol=0.001,verbose=False)

SVC(C=1000000, coef0=0, gamma=0.0001)

In [57]:
#回归
clf = svm.SVR()
SVR(C=1,cache_size=200,coef0=0,degree=3,epsilon=0.1,gamma='auto',kernel='rbf',max_iter=-1,shrinking=True,tol=0.001,verbose=False)

SVR(C=1, coef0=0, gamma='auto')

# 表现和风险评估

In [59]:
#收益率分析

In [62]:
def annualised_sharpe(returns,N=252):
    return np.sqrt(N) * returns.mean() / returns.std()
def equity_sharpe(ticker,start,end):
    pdf = quandl.get(ticker,start_date=start,end_date=end).sort_index()
    pdf['ddaily_ret'] = pdf['Close'].pct_change()
    pdf['excess_daily_ret'] = pdf['daily_ret'] - 0.05/252
    return annualised_sharpe(pd['excess_daily_ret'])

In [64]:
equity_sharpe('WIKI/AAPL',start=datetime.datetime(2011,1,1),end=datetime.datetime(2013,1,1))

LimitExceededError: (Status 429) (Quandl Error QELx06) You have exceeded the API speed limit and your account has temporaly been disabled.  Please contact clientsuccess@nasdaq.com for more information.

# 回测分析
当一种资产的价值下降时，其最高点与最低点之间的长度为回撤

In [65]:
from scipy.stats import norm
def var_cov_var(p,c,mu,sigma):
    alpha = norm.ppf(1-c,mu,sigma)
    return p - p * (alpha + 1)
if __name__ == '__main__':
    start = datetime.datetime(2010,1,1)
    end = datetime.datetime(2014,1,1)
    citi = quandl.get('WIKI/c',start_date = start,end_date = end)
    citi['rets'] = citi['Close'].pct_change()
    p =le6
    c=0.99
    mu = np.mean(citi['rets'])
    sigma = np.std(citi['rets'])
    var =var_cov_var(p,c,mu,sigma)
    print('Value-at-Risk: $%0.2f' % var)

LimitExceededError: (Status 429) (Quandl Error QELx06) You have exceeded the API speed limit and your account has temporaly been disabled.  Please contact clientsuccess@nasdaq.com for more information.